# 乳癌資料庫預測SVM分類
>使用scikit-learn 機器學習套件裡的SVR演算法

* (一)引入函式庫及內建乳癌資料集<br>
引入之函式庫如下<br>
sklearn.datasets: 用來匯入內建之乳癌資料集`datasets.load_breast_cancer()`<br>
sklearn.SVR: 支持向量機回歸分析之演算法<br>
matplotlib.pyplot: 用來繪製影像

# 乳癌資料庫預測 SVM 分類
> 使用 scikit-learn 機器學習套件中的 **SVC（Support Vector Classification）** 進行乳癌資料分類預測

## 說明
本範例使用 sklearn 內建的乳癌資料集，建立 **支持向量機分類模型（SVM / SVC）**，判斷腫瘤是惡性或良性。

### 本範例會用到的工具
- `sklearn.datasets`：載入內建乳癌資料集 `datasets.load_breast_cancer()`
- `sklearn.svm.SVC`：支持向量機**分類**演算法
- `StandardScaler`：做特徵標準化
- `Pipeline`：把標準化與模型訓練串成同一流程，避免資料洩漏
- `train_test_split`：切分訓練集與測試集
- `cross_val_score`：交叉驗證，用來觀察模型穩定性
- `classification_report`：查看 precision、recall、f1-score
- `confusion_matrix`：觀察分類正確與錯誤情況

### 為什麼 SVM 常要先做標準化？
SVM 會根據特徵空間中的距離與邊界來進行分類。  
如果不同特徵的數值尺度差很多，某些特徵可能會主導模型，因此通常會先做標準化。

### 這裡使用的 kernel 與參數概念
- `kernel="rbf"`：讓模型可以學習非線性的分類邊界
- `C`：控制模型對錯分樣本的容忍程度
- `gamma`：控制單一資料點影響範圍

In [19]:
import pandas as pd
from sklearn import svm
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Step1. 下載資料

In [20]:
breast_cancer = datasets.load_breast_cancer()

In [21]:
X = pd.DataFrame(breast_cancer.data, columns = breast_cancer.feature_names)

In [22]:
y = pd.Series(breast_cancer.target, name="target")

In [23]:
display(X.head(), y.head())

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [30]:
print("target names:", breast_cancer.target_names)

target names: ['malignant' 'benign']


In [29]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("-" * 50)
print("類別分布：")
print(y.value_counts())
print("-" * 50)
print("缺失值總數：", X.isnull().sum().sum())

X shape: (569, 30)
y shape: (569,)
--------------------------------------------------
類別分布：
target
1    357
0    212
Name: count, dtype: int64
--------------------------------------------------
缺失值總數： 0


## Step2. 區分訓練集與測試集

這裡使用 `stratify=y`，讓訓練集與測試集都維持原本的類別比例，避免切分後類別分布失衡。

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=0,
    stratify = y
) 

In [32]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (398, 30)
X_test shape: (171, 30)
y_train shape: (398,)
y_test shape: (171,)


## Step3. 建立模型

這裡使用 `Pipeline` 把：
1. `StandardScaler()` 標準化
2. `svm.SVC()` 建立 SVM 分類模型

串成同一個流程。

目前先使用：
- `kernel="rbf"`：可處理非線性邊界
- `C=1`：先使用預設等級的懲罰強度
- `gamma="scale"`：sklearn 常用預設值

In [35]:
clmodel = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", svm.SVC(kernel="rbf", C=1, gamma="scale"))
])

In [36]:
clf_model.fit(X_train, y_train) 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


## Step4. 預測

此處同時會預測訓練集（方便觀察是否過擬合）

In [38]:
y_pred = clf_model.predict(X_test)

In [39]:
y_pred_train = clf_model.predict(X_train)

## Step5. 模型評估

先看：
- 訓練集分數
- 測試集分數
- accuracy

再進一步看：
- confusion matrix
- classification report

In [40]:
print("Train score:", clf_model.score(X_train, y_train))
print("Test score :", clf_model.score(X_test, y_test))

Train score: 0.9949748743718593
Test score : 0.9532163742690059


In [41]:
print("Test accuracy:", accuracy_score(y_test, y_pred))

Test accuracy: 0.9532163742690059


In [45]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Malignant", "Actual Benign"],
    columns=["Predicted Malignant", "Predicted Benign"]
)
display(cm_df)

,Predicted Malignant,Predicted Benign
Actual Malignant,60,4
Actual Benign,4,103


In [46]:
print(classification_report(y_test, y_pred, target_names=breast_cancer.target_names))

              precision    recall  f1-score   support

   malignant       0.94      0.94      0.94        64
      benign       0.96      0.96      0.96       107

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171



## Step6. 交叉驗證（Cross Validation）

這裡只用 **training set** 做 5-fold cross validation，  
目的是觀察模型在訓練資料上的穩定性。

保留下來的 `X_test` / `y_test`，則作為最後的最終驗證。

In [47]:
scores = cross_val_score(clf_model, X_train, y_train, cv=5)

print("Cross-validation scores:", scores)
print("Cross-validation mean :", scores.mean())

Cross-validation scores: [0.975      0.9625     0.975      0.98734177 1.        ]
Cross-validation mean : 0.9799683544303797


## Step7. 嘗試調整 SVM 參數

SVM 常見的重要參數有：
- `C`：越大代表越不想容忍錯分，模型可能更複雜
- `gamma`：越大代表每個點影響範圍越小，邊界可能更彎曲

這裡用 `GridSearchCV` 嘗試找更好的參數組合。

In [48]:
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.1, 0.01, 0.001],
    "svm__kernel": ["rbf"]
}

grid = GridSearchCV(
    estimator=clf_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svm', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svm__C': [0.1, 1, ...], 'svm__gamma': ['scale', 0.1, ...], 'svm__kernel': ['rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 

In [49]:
print("Best params :", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params : {'svm__C': 100, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Best CV score: 0.9849367088607595


In [50]:
best_model = grid.best_estimator_
best_y_pred = best_model.predict(X_test)

print("Best model test accuracy:", accuracy_score(y_test, best_y_pred))

Best model test accuracy: 0.9532163742690059


In [51]:
best_cm = confusion_matrix(y_test, best_y_pred)

best_cm_df = pd.DataFrame(
    best_cm,
    index=["Actual Malignant", "Actual Benign"],
    columns=["Predicted Malignant", "Predicted Benign"]
)
display(best_cm_df)

,Predicted Malignant,Predicted Benign
Actual Malignant,59,5
Actual Benign,3,104


In [52]:
print(classification_report(y_test, best_y_pred, target_names=breast_cancer.target_names))

              precision    recall  f1-score   support

   malignant       0.95      0.92      0.94        64
      benign       0.95      0.97      0.96       107

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171



## Step8. 小結

1. 本範例使用 scikit-learn 提供乳癌資料集建立 SVM 分類模型。
2. 由於 SVM 對特徵尺度敏感，因此先使用 `StandardScaler` 做標準化。
3. 使用 `Pipeline` 可將前處理與模型訓練整合，避免資料洩漏。
4. 除了 accuracy，也應搭配 confusion matrix 與 classification report 來觀察模型表現。
5. 再透過 cross validation 檢查模型穩定性。
6. 最後使用 `GridSearchCV` 調整 `C` 與 `gamma`，找出更合適的參數組合。